# Evaluación Nemotron en FLEURS (es_419/test)

Usa exclusivamente el split español de prueba, en modo streaming, y la ruta de inferencia de producción. No inicia FastAPI ni modifica las métricas de MediaSpeech. El WER de NVIDIA (4,26%) no se afirma como reproducción exacta porque su normalizador completo no está publicado. Fuentes: [FLEURS](https://huggingface.co/datasets/google/fleurs) y [ficha Nemotron](https://huggingface.co/nvidia/nemotron-3.5-asr-streaming-0.6b).

Antes de ejecutar: publica este código en la rama indicada. Cambia `REPO_BRANCH` si corresponde.

In [ ]:
import importlib, json, os, platform, subprocess, sys, time
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), 'Activá Runtime → Change runtime type → GPU'
print(platform.python_version(), torch.__version__, torch.cuda.get_device_name(0))
drive.mount('/content/drive')
DRIVE_BASE = Path('/content/drive/MyDrive/Tesis-subtitles')
CACHE_ROOT = DRIVE_BASE / 'nemotron' / 'cache'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
os.environ['NEMO_CACHE_DIR'] = str(CACHE_ROOT / 'nemo')
os.environ['TORCH_HOME'] = str(CACHE_ROOT / 'torch')
hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
REPO_URL = 'https://github.com/Nacholazabal/subtitle_overlay_fw.git'
REPO_BRANCH = 'feat/perf-profiling'  # Cambiar si publicás la implementación en otra rama
REPO_DIR = Path('/content/subtitle_overlay_fw')
remote = subprocess.run(['git', 'ls-remote', '--heads', REPO_URL, REPO_BRANCH], capture_output=True, text=True)
assert remote.stdout.strip(), f'No encuentro {REPO_BRANCH}; publicá los cambios antes de ejecutar.'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
REQUIRED = ['server/evaluation/fleurs.py', 'server/evaluation/wer_normalization.py', 'server/runtime/nemotron.py']
missing = [name for name in REQUIRED if not (REPO_DIR / name).is_file()]
assert not missing, f'Faltan en la rama remota {missing}; publicá esos archivos antes de ejecutar.'
PROJECT_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
sys.path.insert(0, str(REPO_DIR))
for name in [n for n in list(sys.modules) if n == 'server' or n.startswith('server.')]: del sys.modules[name]
importlib.invalidate_caches()


In [ ]:
# Revisión fija del dataset: permite reanudar aunque cambie 'main' en Hugging Face.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'huggingface_hub', 'soundfile', 'Cython', 'packaging'], check=True)
from datasets import Audio, load_dataset
from server.runtime.nemotron import NEMO_COMMIT, NEMO_REPO, SharedNemotronModel
DATASET_REVISION = '70bb2e84b976b7e960aa89f1c648e09c59f894dd'
DATASET_FINGERPRINT = f'google/fleurs@{DATASET_REVISION}:es_419:test'
def load_fleurs_test():
    ds = load_dataset('google/fleurs', 'es_419', split='test', streaming=True, revision=DATASET_REVISION)
    # Decode one sample at a time inside the evaluator; do not fetch train/other languages.
    return ds.cast_column('audio', Audio(decode=False))
rows_factory = load_fleurs_test
NEMO_DIR = Path('/content/NeMo')
if not NEMO_DIR.exists(): subprocess.run(['git', 'clone', '--filter=blob:none', NEMO_REPO, str(NEMO_DIR)], check=True)
subprocess.run(['git', '-C', str(NEMO_DIR), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(NEMO_DIR), 'checkout', '-q', NEMO_COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{NEMO_DIR}[asr]'], check=True)
sys.path.insert(0, str(NEMO_DIR))
from nemo.collections.asr.inference.factory.pipeline_builder import PipelineBuilder
from server.evaluation.fleurs import FleursEvaluator, fixed_fleurs_config
config = fixed_fleurs_config()
print('FLEURS revision:', DATASET_REVISION, 'project:', PROJECT_COMMIT)
print(json.dumps(config.as_effective_config(), indent=2, ensure_ascii=False))


In [ ]:
from server.evaluation.fleurs import FleursEvaluator
from server.runtime.nemotron import SharedNemotronModel
shared_model = SharedNemotronModel(config)
OUTPUT = Path('/content/drive/MyDrive/Tesis-subtitles/stt_evaluations/fleurs_es_419_test/fleurs-es_419-test__nemotron-560-600-2__v1')
OUTPUT.mkdir(parents=True, exist_ok=True)
evaluator = FleursEvaluator(shared_model, rows_factory, OUTPUT, project_commit=PROJECT_COMMIT, dataset_fingerprint=DATASET_FINGERPRINT, config=config)
print(json.dumps(shared_model.provenance(), indent=2, ensure_ascii=False))
summary = evaluator.run_all()
print((OUTPUT / 'report.md').read_text(encoding='utf-8'))
print('Artefactos:', OUTPUT)